In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull, Delaunay
from inverse_kinematics import IK
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


def is_valid_position(ik_solver, x, y, z, alpha=0):
    result = ik_solver.getRotationAngle((x, y, z), alpha)
    return result is not False

def sample_and_visualize_internal_unreachable_convex(
    x_range=(-55, 55),
    y_range=(-55, 55),
    z_range=(0, 35),
    step=2,
    alpha=0
):
    ik_solver = IK('arm')
    ik_solver.setLinkLength(L1=ik_solver.l1 + 0.75, L4=ik_solver.l4 - 0.15)

    sampled_points = []
    reachable_points = []

    print("Sampling and checking IK reachability...")

    for x in np.arange(x_range[0], x_range[1] + step, step):
        for y in np.arange(y_range[0], y_range[1] + step, step):
            for z in np.arange(z_range[0], z_range[1] + step, step):
                pt = (x, y, z)
                sampled_points.append(pt)
                if is_valid_position(ik_solver, x, y, z, alpha):
                    reachable_points.append(pt)

    print(f"\n🎯 Sampled points: {len(sampled_points)}")
    print(f"✅ Reachable points: {len(reachable_points)}")

    # 使用凸包检查哪些不可达点落在可达点形成的体内
    reachable_array = np.array(reachable_points)
    hull = ConvexHull(reachable_array)
    hull_checker = Delaunay(reachable_array[hull.vertices])

    internal_unreachable = []
    for pt in sampled_points:
        if pt in reachable_points:
            continue
        if hull_checker.find_simplex(pt) >= 0:
            internal_unreachable.append(pt)

    print(f"❌ Internal unreachable points (inside convex hull): {len(internal_unreachable)}")

    # 分别拆分坐标
    rx, ry, rz = zip(*reachable_points)
    if internal_unreachable:
        urx, ury, urz = zip(*internal_unreachable)
    else:
        urx, ury, urz = [], [], []

    # 可视化
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 绘制可达点
    ax.scatter(rx, ry, rz, c='blue', s=10, alpha=0.8, label='Reachable Points (IK ok)')

    # 凸包边界（外壳）
    faces = [reachable_array[simplex] for simplex in hull.simplices]
    ax.add_collection3d(Poly3DCollection(faces, facecolor='cyan', edgecolor='k', alpha=0.2, linewidths=0.3))

    # 可选：显示凸包内的不可达点
    if len(urx) > 0:
        ax.scatter(urx, ury, urz, c='red', s=15, alpha=0.9, label='Unreachable Inside Hull')

    ax.set_xlabel('X (cm)')
    ax.set_ylabel('Y (cm)')
    ax.set_zlabel('Z (cm)')
    ax.set_title('Reachable Convex Hull + Internal Unreachable Points')
    ax.view_init(elev=25, azim=135)
    ax.legend()
    plt.tight_layout()
    plt.show()

    return reachable_points  # ← 新增这一行



if __name__ == "__main__":
    reachable_points = sample_and_visualize_internal_unreachable_convex()


In [ ]:
# Re-import after kernel reset
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Dummy IK function
def is_valid_position(x, y, z):
    return -45 <= x <= 45 and -45 <= y <= 45 and 0 <= z <= 35 and np.linalg.norm([x, y, z]) < 60

# Simulate reachability at bottom Z < 5cm
def simulate_reachability(step=2):
    reachable = []
    unreachable = []
    for x in np.arange(-45, 46, step):
        for y in np.arange(-45, 46, step):
            for z in np.arange(0, 6, step):  # Bottom zone only
                point = (x, y, z)
                if is_valid_position(x, y, z):
                    reachable.append(point)
                else:
                    unreachable.append(point)
    return reachable, unreachable

reachable_bottom, unreachable_bottom = simulate_reachability(step=2)

# Visualization
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

if unreachable_bottom:
    urx, ury, urz = zip(*unreachable_bottom)
    ax.scatter(urx, ury, urz, c='red', s=30, alpha=1.0, label='Unreachable (Z < 5)')

if reachable_bottom:
    rx, ry, rz = zip(*reachable_bottom)
    ax.scatter(rx, ry, rz, c='blue', s=10, alpha=0.5, label='Reachable (Z < 5)')

ax.set_title("Bottom Zone Analysis (Z < 5 cm)")
ax.set_xlabel("X (cm)")
ax.set_ylabel("Y (cm)")
ax.set_zlabel("Z (cm)")
ax.legend()
ax.view_init(elev=-30, azim=135)
plt.tight_layout()
plt.show()


In [ ]:
reachable_array = np.array(reachable_points)

x_min, x_max = reachable_array[:, 0].min(), reachable_array[:, 0].max()
y_min, y_max = reachable_array[:, 1].min(), reachable_array[:, 1].max()
z_min, z_max = reachable_array[:, 2].min(), reachable_array[:, 2].max()
print(x_min, x_max)
print(y_min, y_max)
print(z_min, z_max)

In [ ]:
def get_pick_place_bounds_from_reachable(
    reachable_array,
    x_ratio=0.33,
    yz_buffer=0.15
):
    x = reachable_array[:, 0]
    y = reachable_array[:, 1]
    z = reachable_array[:, 2]

    # X方向切1/3
    x_min, x_max = x.min(), x.max()
    x_span = x_max - x_min
    pick_x = (x_min, x_min + x_ratio * x_span)
    place_x = (x_max - x_ratio * x_span, x_max)

    # YZ方向加buffer
    y_min, y_max = y.min(), y.max()
    z_min, z_max = z.min(), z.max()
    pick_y = place_y = (y_min + yz_buffer * (y_max - y_min), y_max - yz_buffer * (y_max - y_min))
    pick_z = place_z = (z_min + yz_buffer * (z_max - z_min), z_max - yz_buffer * (z_max - z_min))

    pick_box = (pick_x, pick_y, pick_z)
    place_box = (place_x, place_y, place_z)

    return pick_box, place_box
get_pick_place_bounds_from_reachable(reachable_array, x_ratio=0.33, yz_buffer=0.15)

In [ ]:
# 生成 pick 和 place 区域并可视化叠加在可达点图上

def get_pick_place_bounds_from_reachable(
    reachable_array,
    x_ratio=0.33,
    yz_buffer=0.15
):
    x = reachable_array[:, 0]
    y = reachable_array[:, 1]
    z = reachable_array[:, 2]

    # X方向切1/3
    x_min, x_max = x.min(), x.max()
    x_span = x_max - x_min
    pick_x = (x_min, x_min + x_ratio * x_span)
    place_x = (x_max - x_ratio * x_span, x_max)

    # YZ方向加 buffer
    y_min, y_max = y.min(), y.max()
    z_min, z_max = z.min(), z.max()
    pick_y = place_y = (y_min + yz_buffer * (y_max - y_min), y_max - yz_buffer * (y_max - y_min))
    pick_z = place_z = (z_min + yz_buffer * (z_max - z_min), z_max - yz_buffer * (z_max - z_min))

    return (pick_x, pick_y, pick_z), (place_x, place_y, place_z)

def draw_workspace_regions(ax, box, color, label):
    (xr, yr, zr) = box
    x0, x1 = xr
    y0, y1 = yr
    z0, z1 = zr

    corners = np.array([
        [x0, y0, z0], [x1, y0, z0], [x1, y1, z0], [x0, y1, z0],
        [x0, y0, z1], [x1, y0, z1], [x1, y1, z1], [x0, y1, z1],
    ])
    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]
    for i, j in edges:
        ax.plot(
            [corners[i][0], corners[j][0]],
            [corners[i][1], corners[j][1]],
            [corners[i][2], corners[j][2]],
            c=color, linewidth=2, label=label if i == 0 and j == 1 else "")

# 生成区域
pick_box, place_box = get_pick_place_bounds_from_reachable(reachable_array)
rx, ry, rz = reachable_array[:, 0], reachable_array[:, 1], reachable_array[:, 2]

# 绘图
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(rx, ry, rz, c='blue', s=10, alpha=0.5, label='Reachable Points')
draw_workspace_regions(ax, pick_box, 'green', 'Pick Region')
draw_workspace_regions(ax, place_box, 'red', 'Place Region')

ax.set_xlabel("X (cm)")
ax.set_ylabel("Y (cm)")
ax.set_zlabel("Z (cm)")
ax.set_title("Auto-defined Pick & Place Regions from Reachable Space")
ax.legend()
plt.tight_layout()
plt.show()
